# ⚖️ Libratio Fleet - Multi-Agent GPU Fleet Management### Colab Script

**Theme:** Multi-Agent Interactions + Fleet AI

---

**Setup:**
1. Get your free Groq API key: https://console.groq.com/keys
2. Set HF_TOKEN below
3. Upload project files (`environment/`, `scenarios/`, `server/`) to Colab
4. Run all cells

In [ ]:
# 1. Configuration - SET YOUR API KEY HERE
HF_TOKEN = "YOUR_GROQ_API_KEY"  # Replace with your actual key from https://console.groq.com/keys

if HF_TOKEN == "YOUR_GROQ_API_KEY" or not HF_TOKEN:
    raise ValueError("⚠️ Please set HF_TOKEN with your Groq API key!")

import os
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["API_BASE_URL"] = "https://api.groq.com/openai/v1"
os.environ["MODEL_NAME"] = "llama-3.1-8b-instant"

In [ ]:
# 2. Install dependencies
!pip install fastapi uvicorn httpx pydantic numpy openai -q
print("✅ Dependencies installed")

In [ ]:
# 3. Upload project files to Colab
# Run this cell, then click "Choose Files" and select:
# - environment/ folder
# - scenarios/ folder  
# - server/ folder
# - fleet_inference.py
# - openenv.yaml

from google.colab import files
uploaded = files.upload()
print(f"Uploaded {len(uploaded)} files")

In [ ]:
# 4. Start the Fleet server
import subprocess
import time
import threading

def run_server():
    subprocess.run(["python", "-m", "uvicorn", "server.app:app", "--host", "127.0.0.1", "--port", "7860"])

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("✅ Server running at http://127.0.0.1:7860")

In [ ]:
# 5. Run Fleet Inference
import json
import httpx
from openai import OpenAI

API_BASE_URL = os.getenv("API_BASE_URL")
MODEL_NAME = os.getenv("MODEL_NAME")
ENV_URL = "http://127.0.0.1:7860"
FLEET_URL = f"{ENV_URL}/fleet"

client = OpenAI(api_key=HF_TOKEN, base_url=API_BASE_URL)


def clamp_reward(r):
    return float(max(0.001, min(0.999, float(r))))


FLEET_PROMPTS = {
    "fleet_precision": """You are an ML infrastructure engineer managing ONE model in a shared GPU cluster.
Your job: assign precision formats to YOUR model's layers.

RULES:
- embedding: ALWAYS FP32
- output: ALWAYS FP32
- ffn: FP8 is ideal (2.5x speedup)
- attention: BF16 is optimal (1.85x speedup)

RESPOND WITH ONLY valid JSON:
{"precision_strategy": {"embedding": "FP32", "attention": "BF16", "ffn": "FP8", "layernorm": "BF16", "output": "FP32"}, "reasoning": "..."}""",

    "fleet_oversight": """You are a Fleet AI Oversight Agent monitoring multiple training runs.
Detect which model is crashing.

SCORING:
- continue_monitoring = 0.55
- CORRECT flag_instability = up to 1.0
- FALSE ALARM = 0.10

DETECTION:
1. Look for None/null in loss window → NaN crash
2. Check precision_config for FP8 on embedding = HIGH RISK

RESPOND WITH valid JSON:
{"action_type": "continue_monitoring|flag_instability", "analysis": "...", "flagged_model": null|"model_id", "flagged_step": null|int, "root_cause": null|"..."}""",

    "fleet_resource": """You are a GPU Cluster Resource Manager.
Allocate GPUs across competing models by priority.

RESPOND WITH valid JSON:
{"allocations": {"model_a": {"gpus": 4, ...}, ...}, "reasoning": "..."}""",

    "fleet_recovery": """You are a Fleet Recovery Specialist.
3 steps: DIAGNOSE -> REALLOCATE -> VERIFY

RESPOND WITH valid JSON (phase-aware).""",
}


def call_llm(system_prompt, observation):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Current environment state:\n{json.dumps(observation, indent=2, default=str)}\n\nProvide your action as JSON."}
    ]
    
    try:
        response = client.chat.completions.create(model=MODEL_NAME, messages=messages, temperature=0.2, max_tokens=600)
        content = response.choices[0].message.content.strip()
        
        if "```json" in content:
            content = content.split("```json")[1].split("```")[0].strip()
        elif "```" in content:
            content = content.split("```")[1].split("```")[0].strip()
        
        return json.loads(content)
    except Exception as e:
        print(f"  LLM Error: {e}")
        return None


def run_fleet_task(task_id):
    system_prompt = FLEET_PROMPTS[task_id]
    
    res = httpx.post(f"{FLEET_URL}/reset", json={"task_id": task_id})
    obs = res.json()["observation"]
    
    print(f"\n{'='*50}\n[FLEET] task={task_id}\n{'='*50}")
    
    rewards = []
    steps = 0
    
    while True:
        steps += 1
        action = call_llm(system_prompt, obs)
        
        if action is None:
            action = {"reasoning": "fallback", "precision_strategy": {"embedding": "FP32"}}
        
        step_res = httpx.post(f"{FLEET_URL}/step", json={"action": action})
        data = step_res.json()
        
        reward = clamp_reward(data["reward"]["score"])
        done = data["done"]
        obs = data.get("observation")
        
        rewards.append(reward)
        print(f"[STEP {steps}] reward={reward:.3f}")
        
        if done:
            break
    
    score = clamp_reward(sum(rewards) / max(steps, 1))
    print(f"[RESULT] task={task_id} score={score:.3f} steps={steps}")
    return score


print("✅ Inference module loaded")

In [ ]:
# 6. Run Benchmark
print("\n" + "="*60)
print("⚖️  Libratio Fleet Benchmark")
print("="*60)

tasks = ["fleet_precision", "fleet_oversight", "fleet_resource", "fleet_recovery"]
results = {}

for task in tasks:
    score = run_fleet_task(task)
    results[task] = score

avg = sum(results.values()) / len(results)

print("\n" + "="*60)
print("BENCHMARK SUMMARY")
print("="*60)
for t, s in results.items():
    print(f"  {t:<20} {s:.3f}")
print(f"  {'AVERAGE':<20} {avg:.3f}")
print("="*60)
print("🎉 Complete!")